# exp5 — candidate-only Kaggle check

**Timestamp:** 2026-10-24 14:45 MSK  
**Intent:** Check whether the candidate-only, early-MAP-tuned RRF can improve the designated Kaggle result before any learned reranker. No target views from weeks31–42 were used.

## Method

Candidate pipeline: history weeks11–30, target schedule weeks31–42, tripartite PPR + LightFM WARP + ALS + repeat/popularity + time/channel/metadata/schedule propensity. Fusion weights were tuned on early validation only. The candidate pool gate remains separate: main Recall@500 is below the user-confirmed 0.80 threshold, so no binary LightGBM or LGBMRanker was trained.


In [1]:
! .venv-exp5/bin/python artifacts/scripts/exp5_candidate_recall.py expanded9 final

[exp5] history rows=3,847,696; users=4,895; target catalog=6,632
[exp5] eval users with strong target=1,260; relevance users >=.5=0, >=.8=0
  [graph] пользователей=4895, программ=7884 (train=4363, пул без просмотров в train=3521), атрибутов=264, программ без атрибутов=0
[exp5] final candidates rows=2,715,501; users=1,260; mean pool/user=2155.2
[exp5] saved=/Users/davidkk/Kaggle_Case/data/exp5_candidates_final_expanded9.parquet; target shows=6,632; training history<=30; no target views were read

In [2]:
! .venv/bin/python artifacts/scripts/exp5_make_candidate_submission.py

[exp5] output=/Users/davidkk/Kaggle_Case/data/submission_exp5_candidate_map_tuned.csv; rows=1260; predictions_per_user=5; all IDs in target schedule
[exp5] unique scheduled shows in output=125
[exp5] source weights: {'repeat': 4.0, 'lightfm': 1.0, 'ppr': 8.0, 'popularity': 1.0, 'tv_slot': 1.0, 'user_channel': 0.5, 'user_metadata': 1.0, 'als': 2.0, 'user_time_slot': 0.0, 'user_channel_dow': 0.0, 'user_channel_hour': 0.0, 'airing_popularity': 2.0, 'prop_max': 0.5, 'prop_mean': 0.0, 'prop_sum': 16.0, 'prop_logsumexp': 16.0}

In [3]:
! kaggle competitions submit -c sweettv-tv-program-recommender -f data/submission_exp5_candidate_map_tuned.csv -m "exp5 candidate-only tripartite MAP5-tuned RRF"

99 submissions remaining today.
Successfully submitted to sweet.tv - TV program recommender

In [4]:
! kaggle competitions submit -c sweettv-tv-program-recommender -f data/submission_exp5_candidate_lightfm_threshold_08.csv -m "exp5 candidate-only LightFM history threshold 0.8"

98 submissions remaining today.
Successfully submitted to sweet.tv - TV program recommender

In [5]:
! kaggle competitions submit -c sweettv-tv-program-recommender -f data/submission_exp5_candidate_lightfm_threshold_05.csv -m "exp5 candidate-only LightFM history threshold 0.5"

97 submissions remaining today.
Successfully submitted to sweet.tv - TV program recommender

In [6]:
! kaggle competitions submissions -c sweettv-tv-program-recommender | head -5

ref  fileName                                            date                        description                                                                            status                     publicScore  privateScore
57593488  submission_exp5_candidate_lightfm_threshold_05.csv  2026-10-24 12:01:19.150000  exp5 candidate-only LightFM history threshold 0.5  SubmissionStatus.COMPLETE  0.38093  0.36288
57593473  submission_exp5_candidate_lightfm_threshold_08.csv  2026-10-24 12:01:01.517000  exp5 candidate-only LightFM history threshold 0.8  SubmissionStatus.COMPLETE  0.40581  0.38178
57593168  submission_exp5_candidate_map_tuned.csv             2026-10-24 11:48:06.153000  exp5 candidate-only tripartite MAP5-tuned RRF      SubmissionStatus.COMPLETE  0.41874  0.39477
57554720  submission_exp4_k10.csv                             2026-10-23 08:29:07.453000  exp4 k10 (previous designated final)                SubmissionStatus.COMPLETE  0.34586  0.32383

## Result interpretation

The candidate-only submission `57593168` reached public/private MAP@5 0.41874/0.39477, beating the previous exp4_k10 result 0.34586/0.32383 on both halves. Threshold-specific LightFM variants scored lower: 0.40581/0.38178 at 0.8 and 0.38093/0.36288 at 0.5. Therefore the MAP-tuned threshold-free LightFM candidate RRF remains the current best Kaggle file; root `submission.csv` was not overwritten.

This does not authorize the reranker: main validation Recall@500 is still below 0.80. No LightGBM model was trained, and no claim of reaching the leader public score (~0.453) is made.
